# YOLO26 Scratch Segmentation Training

This notebook trains a YOLO segmentation model from the converted dataset.

Expected dataset format:

```text
scratch_yolo_seg/
  train/
    images/
    labels/
  valid/
    images/
    labels/
  test/
    images/
    labels/
  data.yaml
```

Upload either `scratch_yolo_seg/` or `scratch_yolo_seg.zip` to Colab before running the cells.

## 1. Install Ultralytics

Use the latest Ultralytics package so YOLO26 model names are available.

In [ ]:
!python -m pip install -q -U ultralytics

from pathlib import Path
import shutil
import zipfile

import yaml
from ultralytics import YOLO

print('Ultralytics is ready')

## 2. Dataset Settings

The notebook looks for the dataset in common Colab upload locations. If it finds a zip file, it extracts it automatically.

In [ ]:
DATASET_NAME = 'scratch_yolo_seg'

candidate_dirs = [
    Path('/content') / DATASET_NAME,
    Path('/content/data') / DATASET_NAME,
    Path('/content/drive/MyDrive/Surface-Scratch-Detection/data') / DATASET_NAME,
    Path('/content/drive/MyDrive/Surface-Scratch-Detection') / DATASET_NAME,
]

candidate_zips = [
    Path('/content') / f'{DATASET_NAME}.zip',
    Path('/content/data') / f'{DATASET_NAME}.zip',
    Path('/content/drive/MyDrive/Surface-Scratch-Detection/data') / f'{DATASET_NAME}.zip',
    Path('/content/drive/MyDrive/Surface-Scratch-Detection') / f'{DATASET_NAME}.zip',
]

def find_dataset_root() -> Path:
    for path in candidate_dirs:
        if path.is_dir():
            return path

    for zip_path in candidate_zips:
        if not zip_path.is_file():
            continue

        extract_root = Path('/content')
        print(f'Extracting {zip_path} -> {extract_root}')
        with zipfile.ZipFile(zip_path, 'r') as archive:
            archive.extractall(extract_root)

        for path in candidate_dirs:
            if path.is_dir():
                return path

    raise FileNotFoundError(
        'Dataset not found. Upload scratch_yolo_seg/ or scratch_yolo_seg.zip to /content.'
    )

DATASET_ROOT = find_dataset_root()
DATA_YAML = DATASET_ROOT / 'data.yaml'

print('Dataset root:', DATASET_ROOT)
print('data.yaml:', DATA_YAML)

## 3. Check YOLO Dataset Structure

In [ ]:
required_dirs = [
    DATASET_ROOT / 'train' / 'images',
    DATASET_ROOT / 'train' / 'labels',
    DATASET_ROOT / 'valid' / 'images',
    DATASET_ROOT / 'valid' / 'labels',
    DATASET_ROOT / 'test' / 'images',
    DATASET_ROOT / 'test' / 'labels',
]

missing = [path for path in required_dirs if not path.is_dir()]
if missing:
    raise FileNotFoundError('Missing dataset folders:\n' + '\n'.join(str(path) for path in missing))
if not DATA_YAML.is_file():
    raise FileNotFoundError(f'data.yaml not found: {DATA_YAML}')

for split in ('train', 'valid', 'test'):
    image_dir = DATASET_ROOT / split / 'images'
    label_dir = DATASET_ROOT / split / 'labels'
    images = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}])
    labels = sorted(label_dir.glob('*.txt'))
    missing_labels = [p.name for p in images if not (label_dir / f'{p.stem}.txt').is_file()]
    positive_labels = [p for p in labels if p.stat().st_size > 0]
    print(
        f'{split}: images={len(images)} labels={len(labels)} '
        f'positive_labels={len(positive_labels)} missing_labels={len(missing_labels)}'
    )
    if missing_labels:
        raise RuntimeError(f'{split} has missing labels. Example: {missing_labels[:5]}')

with DATA_YAML.open('r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)

print('\ndata.yaml:')
print(yaml.safe_dump(data_config, sort_keys=False))

## 4. Train YOLO26 Segmentation

Use `yolo26n-seg.pt` for this dataset because labels are YOLO polygon instance-segmentation labels. Do not use `yolo26n-sem.pt` for this converted dataset.

In [ ]:
MODEL = 'yolo26n-seg.pt'
PROJECT = '/content/yolo_scratch_runs'
RUN_NAME = 'scratch_yolo26n_seg'

EPOCHS = 100
IMGSZ = 512
BATCH = 16
PATIENCE = 30
WORKERS = 2

model = YOLO(MODEL)

results = model.train(
    data=str(DATA_YAML),
    task='segment',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    plots=True,
    save=True,
    device=0,
)

RUN_DIR = Path(PROJECT) / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'

print('Run dir:', RUN_DIR)
print('Best checkpoint:', BEST_PT, BEST_PT.exists())
print('Last checkpoint:', LAST_PT, LAST_PT.exists())

## 5. Validate Best Checkpoint

In [ ]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(
    data=str(DATA_YAML),
    task='segment',
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    plots=True,
    device=0,
)

print(metrics)

## 6. Download Checkpoints

In [ ]:
from google.colab import files

if BEST_PT.is_file():
    files.download(str(BEST_PT))
if LAST_PT.is_file():
    files.download(str(LAST_PT))